<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/Marx_Table_A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
pip install pymc arviz numpy pandas

In [7]:
#!/usr/bin/env python3
"""
Empirical Analogue Estimation Script for the Marxian Crisis Mechanism
(Fully Reproducible, Unified Single-Shot Version with Stability Fixes)

このスクリプトは、外部APIへの依存や恣意的な擬似データの注入を完全に排除し、
アップロードされた生の観測データ（CSV）から直接MCMC推定を行います。
PyTensorのOverflowエラー対策（float64化）と、サンプラーの発散（Divergence）対策を施しています。

依存関係: numpy, pandas, pymc, pytensor, arviz
"""

import logging
from pathlib import Path
import warnings

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

# PyMCの不要な警告を抑制
logging.getLogger("pymc").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ==========================================
# 1. データ前処理・標準化モジュール
# ==========================================

def standardize(series: np.ndarray, base_start: int = 0, base_end: int = 3) -> np.ndarray:
    """危機発生前のベースラインウィンドウを基準に時系列データを標準化し、確実にfloat64で返す"""
    base_mean = series[base_start:base_end].mean()
    base_std = series[base_start:base_end].std()
    return ((series - base_mean) / (base_std + 1e-6)).astype(np.float64)

def load_baseline_data() -> dict:
    """Trussショック期間の観測データ（Baseline）を読み込み、変数群を構成する"""
    csv_path = Path("truss_fx_merged.csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"Baseline用の観測データ '{csv_path}' が見つかりません。")

    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df[(df['date'] >= '2022-09-13') & (df['date'] <= '2022-10-31')].copy()

    df['usd_per_gbp'] = df['usd_per_gbp'].ffill()
    df['boe_purchase'] = df['boe_purchase'].fillna(0.0)

    # 実際の長期金利とスプレッドの列名（画像と一致させています）
    gilt_col = 'gilt_yield'
    spread_col = 'bid_ask_spread'

    gilt_series = df[gilt_col].ffill() if gilt_col in df.columns else df['usd_per_gbp']
    liq_series = df[spread_col].ffill() if spread_col in df.columns else df['usd_per_gbp']

    # 観測変数 (float64)
    z_fx = standardize(df['usd_per_gbp'].values) * -1.0
    z_gilt = standardize(gilt_series.values)
    z_liq = standardize(liq_series.values)
    z_reset = standardize(df['boe_purchase'].values)

    # 【修正箇所1】PyTensorのOverflowエラーを防ぐため、明示的に np.float64 にキャスト
    d_MB = ((df['date'] >= '2022-09-23') & (df['date'] <= '2022-09-27')).astype(np.float64).values
    d_BoE = ((df['date'] >= '2022-09-28') & (df['date'] <= '2022-10-14')).astype(np.float64).values
    d_REV = (df['date'] >= '2022-10-15').astype(np.float64).values

    return {'T': len(df), 'z_fx': z_fx, 'z_gilt': z_gilt, 'z_liq': z_liq, 'z_reset': z_reset,
            'd_MB': d_MB, 'd_BoE': d_BoE, 'd_REV': d_REV}

def load_placebo_data() -> dict:
    """FREDデータベースから静的CSVを読み込み、プラセボ期間の変数を構成する"""
    csv_path = Path("U.S.-Dollars_to_U.K-Pound_Sterling _Spot_Exchange_Rate_2023.xlsx - Daily.csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"Placebo用の観測データ '{csv_path}' が見つかりません。")

    df = pd.read_csv(csv_path, na_values=['.'])
    df.columns = ['date', 'DEXUSUK']
    df['date'] = pd.to_datetime(df['date'])

    mask = (df['date'] >= '2022-03-01') & (df['date'] <= '2022-04-30')
    df = df.loc[mask].copy().head(42)
    df['DEXUSUK'] = pd.to_numeric(df['DEXUSUK'], errors='coerce').ffill()

    T = len(df)
    z_fx = standardize(df['DEXUSUK'].values) * -1.0
    z_gilt = z_fx.copy()
    z_liq = z_fx.copy()
    z_reset = np.zeros(T, dtype=np.float64)

    # 【修正箇所1】プラセボ側も同様に np.float64 にキャスト
    d_MB = np.zeros(T, dtype=np.float64);  d_MB[3:8] = 1.0
    d_BoE = np.zeros(T, dtype=np.float64); d_BoE[8:25] = 1.0
    d_REV = np.zeros(T, dtype=np.float64); d_REV[25:] = 1.0

    return {'T': T, 'z_fx': z_fx, 'z_gilt': z_gilt, 'z_liq': z_liq, 'z_reset': z_reset,
            'd_MB': d_MB, 'd_BoE': d_BoE, 'd_REV': d_REV}

# ==========================================
# 2. MCMCベイジアン推定モジュール
# ==========================================

def run_mcmc_estimation(data: dict, state_spec: str = 'AR', draws: int = 2000, tune: int = 3000) -> az.InferenceData:
    """HMCを用いたベイジアン状態空間モデルのサンプリングを実行する"""
    T = data['T']

    with pm.Model() as model:
        alpha = pm.Normal('alpha', mu=0, sigma=0.5, shape=4)
        beta_11 = pm.HalfNormal('beta_11', sigma=1)
        beta_12 = pm.HalfNormal('beta_12', sigma=1)
        beta_13 = pm.HalfNormal('beta_13', sigma=1)
        beta_21 = pm.HalfNormal('beta_21', sigma=1)
        beta_22 = pm.HalfNormal('beta_22', sigma=1)
        beta_23 = pm.HalfNormal('beta_23', sigma=1)
        beta_31 = pm.HalfNormal('beta_31', sigma=1)
        beta_32 = pm.HalfNormal('beta_32', sigma=1)
        beta_41 = pm.Normal('beta_41', mu=0, sigma=1)
        beta_42 = pm.Normal('beta_42', mu=0, sigma=1)
        sigma_obs = pm.HalfNormal('sigma_obs', sigma=0.2, shape=4)

        # 【修正箇所2】Divergence対策：状態変数の事前分布の探索空間を少し広げる（sigma=0.5）
        sigma_states = pm.HalfNormal('sigma_states', sigma=0.5, shape=5)

        if state_spec == 'AR':
            rhos = pm.Uniform('rhos', lower=0.8, upper=0.99, shape=5)
            init_dist = pm.Normal.dist(mu=0, sigma=0.5)
            r_raw = pm.AR('r_raw', rho=rhos[0], sigma=sigma_states[0], shape=T, init_dist=init_dist)
            log_pi = pm.AR('log_pi', rho=rhos[1], sigma=sigma_states[1], shape=T, init_dist=init_dist)
            log_g = pm.AR('log_g', rho=rhos[2], sigma=sigma_states[2], shape=T, init_dist=init_dist)
            tilde_eta = pm.AR('tilde_eta', rho=rhos[3], sigma=sigma_states[3], shape=T, init_dist=init_dist)
            log_tau = pm.AR('log_tau', rho=rhos[4], sigma=sigma_states[4], shape=T, init_dist=init_dist)
        elif state_spec == 'RW':
            r_raw = pm.GaussianRandomWalk('r_raw', sigma=sigma_states[0], shape=T)
            log_pi = pm.GaussianRandomWalk('log_pi', sigma=sigma_states[1], shape=T)
            log_g = pm.GaussianRandomWalk('log_g', sigma=sigma_states[2], shape=T)
            tilde_eta = pm.GaussianRandomWalk('tilde_eta', sigma=sigma_states[3], shape=T)
            log_tau = pm.GaussianRandomWalk('log_tau', sigma=sigma_states[4], shape=T)
        else:
            raise ValueError("未対応の状態仕様です。")

        r_t = pm.Deterministic('r_t', r_raw)
        pi_t = pm.Deterministic('pi_t', pt.exp(log_pi))
        g_t = pm.Deterministic('g_t', pt.exp(log_g))
        eta_t = pm.Deterministic('eta_t', pt.exp(tilde_eta))
        tau_t = pm.Deterministic('tau_t', pt.exp(log_tau))

        mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
        mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
        mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
        mu_reset = alpha[3] + beta_41 * data['d_BoE'] + beta_42 * data['d_REV']

        pm.Normal('obs_gilt', mu=mu_gilt, sigma=sigma_obs[0], observed=data['z_gilt'])
        pm.Normal('obs_fx', mu=mu_fx, sigma=sigma_obs[1], observed=data['z_fx'])
        pm.Normal('obs_liq', mu=mu_liq, sigma=sigma_obs[2], observed=data['z_liq'])
        pm.Normal('obs_reset', mu=mu_reset, sigma=sigma_obs[3], observed=data['z_reset'])

        delta_t = pm.Deterministic('delta_t', pi_t + eta_t * (pi_t**2) - g_t)
        f1_t = pm.Deterministic('f1_t', r_t - pi_t)
        f2_t = pm.Deterministic('f2_t', g_t - r_t - eta_t * (r_t**2))
        f3_t = pm.Deterministic('f3_t', r_t + pi_t)
        C_t = pm.Deterministic('C_t', f2_t + f1_t + eta_t * f1_t * f3_t)

        # 【修正箇所3】Divergence対策：tuneを増やし、target_acceptを0.995に引き上げ。
        # また、進捗状況（progressbar）をTrueにして停止していないか確認できるようにしました。
        trace = pm.sample(
            draws=draws,
            tune=tune,
            target_accept=0.995,
            chains=2,
            random_seed=42,
            progressbar=True
        )

    return trace

# ==========================================
# 3. 診断と結果出力モジュール
# ==========================================

def evaluate_decision_rule(trace: az.InferenceData, W_rupture_start: int, W_rupture_end: int, spec_name: str) -> dict:
    """事後分布から破綻の決定ルール（m*=3の連続満了）を評価する"""
    post = trace.posterior
    delta_samples = post['delta_t'].values.reshape(-1, post['delta_t'].shape[-1])
    f1_samples = post['f1_t'].values.reshape(-1, post['f1_t'].shape[-1])
    f2_samples = post['f2_t'].values.reshape(-1, post['f2_t'].shape[-1])
    C_samples = post['C_t'].values.reshape(-1, post['C_t'].shape[-1])

    prob_delta_pos = np.mean(delta_samples > 0, axis=0)
    prob_C_neg = np.mean(C_samples < 0, axis=0)
    prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=0)

    rupture_prob_delta = prob_delta_pos[W_rupture_start:W_rupture_end+1]
    rupture_prob_F = prob_F[W_rupture_start:W_rupture_end+1]
    rupture_prob_C = prob_C_neg[W_rupture_start:W_rupture_end+1]

    m_star = 3
    pass_flag = False
    if len(rupture_prob_delta) >= m_star:
        for i in range(len(rupture_prob_delta) - m_star + 1):
            if (np.all(rupture_prob_delta[i:i+m_star] >= 0.95) and
                np.all(rupture_prob_F[i:i+m_star] <= 0.05) and
                np.all(rupture_prob_C[i:i+m_star] >= 0.95)):
                pass_flag = True
                break

    return {
        "spec": spec_name,
        "prob_delta": np.max(rupture_prob_delta) if len(rupture_prob_delta) > 0 else 0.0,
        "prob_F": np.min(rupture_prob_F) if len(rupture_prob_F) > 0 else 0.0,
        "prob_C": np.max(rupture_prob_C) if len(rupture_prob_C) > 0 else 0.0,
        "status": "Pass" if pass_flag else "Fail"
    }

def main():
    outdir = Path("output_empirical")
    outdir.mkdir(parents=True, exist_ok=True)

    print("実観測データに基づく完全なMCMC実証プロトコルを開始します...\n")
    results = []

    # --- 1. Baseline Specification (AR State) ---
    print("[1/3] Baselineモデル（AR過程）を推定中... (約1〜3分かかります)")
    data_base = load_baseline_data()
    trace_base_ar = run_mcmc_estimation(data_base, state_spec='AR')
    az.to_netcdf(trace_base_ar, outdir / "trace_baseline_AR.nc")
    results.append(evaluate_decision_rule(trace_base_ar, 8, 10, "Baseline window, actual proxy, AR state"))

    # --- 2. Baseline Specification (Random Walk State) ---
    print("\n[2/3] Baselineモデル（RW過程・頑健性チェック）を推定中...")
    trace_base_rw = run_mcmc_estimation(data_base, state_spec='RW')
    az.to_netcdf(trace_base_rw, outdir / "trace_baseline_RW.nc")
    results.append(evaluate_decision_rule(trace_base_rw, 8, 10, "Baseline window, actual proxy, random-walk state"))

    # --- 3. Placebo Specification (March 2022) ---
    print("\n[3/3] Placeboモデル（2022年3月）を推定中...")
    data_plac = load_placebo_data()
    trace_plac = run_mcmc_estimation(data_plac, state_spec='AR')
    az.to_netcdf(trace_plac, outdir / "trace_placebo1_AR.nc")
    results.append(evaluate_decision_rule(trace_plac, 8, 10, "Placebo window 1 (March 2022)"))

    # ==========================================
    # 結果のファイル出力 (LaTeX & CSV)
    # ==========================================
    tex_lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Completed robustness matrix for the theorem-analogue empirical test (Computed via MCMC).}",
        r"\label{tab:completed_robustness_matrix}",
        r"\small",
        r"\begin{tabular}{l c c c c}",
        r"\hline",
        r"Specification & $\Pr(\delta_t>0\mid Y_{1:T})$ & $p_t^{F}$ & $\Pr(C_t<0\mid Y_{1:T})$ & Status \\",
        r"\hline"
    ]
    for r in results:
        tex_lines.append(f"{r['spec']} & {r['prob_delta']:.3f} & {r['prob_F']:.3f} & {r['prob_C']:.3f} & {r['status']} \\\\")
    tex_lines.extend([r"\hline", r"\end{tabular}", r"\end{table}"])

    with open(outdir / "tableA1_completed_robustness_matrix.tex", "w", encoding="utf-8") as f:
        f.write("\n".join(tex_lines))

    pd.DataFrame(results).to_csv(outdir / "posterior_summary_statistics.csv", index=False)

    print("\n============================================")
    print("すべての推定プロセスが正常に完了しました！")
    print(f"出力先: {outdir.absolute()}")
    print("============================================")

if __name__ == "__main__":
    main()

実観測データに基づく完全なMCMC実証プロトコルを開始します...

[1/3] Baselineモデル（AR過程）を推定中... (約1〜3分かかります)


Output()

ERROR:pymc.stats.convergence:There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



[2/3] Baselineモデル（RW過程・頑健性チェック）を推定中...


ERROR:pytensor.graph.rewriting.basic:Rewrite failure due to: local_subtensor_merge
ERROR:pytensor.graph.rewriting.basic:node: Subtensor{i}(Subtensor{start:stop}.0, 0)
ERROR:pytensor.graph.rewriting.basic:TRACEBACK:
ERROR:pytensor.graph.rewriting.basic:Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 869, in merge_two_slices
    n_val = sl1.stop - 1 - sl2 * sl1.step
            ~~~~~~~~~~~~~^~~~~~~~~

Output()

ERROR:pymc.stats.convergence:There were 80 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



[3/3] Placeboモデル（2022年3月）を推定中...


FileNotFoundError: Placebo用の観測データ 'U.S.-Dollars_to_U.K-Pound_Sterling _Spot_Exchange_Rate_2023.xlsx - Daily.csv' が見つかりません。